# Actividad 3: Aplicación de algoritmos de aprendizaje supervisado con PySpark

**Materia:** Análisis de grandes volúmenes de datos  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Autor:** Jonathan Javier Monsalve Giraldo (A01840272)  
**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 24 de mayo de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025  
**Modalidad:** Individual

## Objetivo

Aplicar un algoritmo de aprendizaje supervisado en PySpark MLlib sobre una muestra M' derivada de la muestra estratificada M construida en la Etapa 2 del proyecto del equipo. El problema elegido es regresión sobre `fare_amount` (tarifa base registrada del viaje), enmarcado como un modelo operativo de auditoría tarifaria a partir de variables del registro del viaje.

## Estructura del notebook

1. **Introducción**: aprendizaje supervisado, algoritmos representativos y los disponibles en PySpark MLlib.
2. **Selección de los datos**: reconstrucción compacta de M (recap de la Etapa 2) y construcción de la muestra individual M'.
3. **Preparación del conjunto de entrenamiento y prueba**: definición del problema, features y partición estratificada.
4. **Construcción de modelos**: pipeline, entrenamiento, evaluación e interpretación de dos modelos supervisados.

### Nota para el profesor

La Sección 2.0 reproduce de forma compacta la muestra M de la Etapa 2. El aporte individual inicia en la **Sección 2.1**, con la construcción de M', la partición train/test y el modelado supervisado.

## 1. Introducción

### 1.1 Aprendizaje supervisado

El aprendizaje supervisado es el paradigma en el que un modelo aprende una relación entre un conjunto de variables predictoras y una variable objetivo conocida en los datos de entrenamiento, para después aplicar esa relación a observaciones nuevas. Cada fila del conjunto de entrenamiento incluye tanto las predictoras como la respuesta etiquetada, y la calidad del modelo se evalúa sobre datos no usados durante el ajuste para estimar su capacidad de generalización.

Se distinguen dos grandes tareas según la naturaleza de la variable objetivo. En **regresión** la respuesta es continua, como el monto de una tarifa, la demanda esperada o la temperatura. En **clasificación** la respuesta es categórica, binaria o multiclase, como fraude vs no fraude, especie de una flor o tipo de pago. Esta actividad aborda un problema de regresión.

### 1.2 Algoritmos representativos

La literatura agrupa los algoritmos supervisados en familias con supuestos y compromisos distintos:

- **Modelos lineales** (regresión lineal, regresión logística): coeficientes interpretables, supuesto de linealidad, sensibles a multicolinealidad y a la escala de los predictores. Útiles como baseline y cuando la relación esperada es aproximadamente lineal.
- **Árboles de decisión**: particiones recursivas del espacio de features, capturan no linealidades e interacciones, fáciles de interpretar como reglas; un árbol único tiende a sobreajustar si la profundidad crece.
- **Ensembles de árboles** (Random Forest, Gradient Boosted Trees): combinan muchos árboles para reducir varianza o sesgo. Suelen rendir bien en datos tabulares y exponen importancia de variables, a costa de menor interpretabilidad y mayor costo de cómputo.
- **Máquinas de vectores de soporte (SVM)**: separan clases maximizando el margen, eficaces con datos de alta dimensión y kernels para relaciones no lineales; menos comunes en Big Data por su costo cuadrático.
- **Perceptrón multicapa (MLP)**: redes feedforward capaces de aprender relaciones complejas; requieren más datos, tuning cuidadoso y aportan poca interpretabilidad directa.
- **Naive Bayes**: clasificador probabilístico basado en independencia condicional entre features; rápido y útil en problemas de conteo o texto.

### 1.3 Disponibles en PySpark MLlib

PySpark expone los algoritmos anteriores a través del módulo moderno `pyspark.ml`, basado en DataFrames y organizado bajo el patrón Estimator-Transformer-Pipeline. Un *Estimator* aprende parámetros con `fit()` (por ejemplo, `LinearRegression`, `RandomForestClassifier`); un *Transformer* aplica una transformación con `transform()` (por ejemplo, un modelo ya entrenado o un `VectorAssembler`); un *Pipeline* encadena pasos para garantizar que el mismo preprocesamiento aprendido en train se aplique a test, evitando fuga de información.

| Tarea | Submódulo | Algoritmos disponibles |
|---|---|---|
| Regresión | `pyspark.ml.regression` | `LinearRegression`, `GeneralizedLinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor`, `GBTRegressor`, `IsotonicRegression`, `AFTSurvivalRegression`, `FMRegressor` |
| Clasificación | `pyspark.ml.classification` | `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`, `GBTClassifier`, `MultilayerPerceptronClassifier`, `NaiveBayes`, `LinearSVC`, `OneVsRest`, `FMClassifier` |
| Evaluación | `pyspark.ml.evaluation` | `RegressionEvaluator`, `BinaryClassificationEvaluator`, `MulticlassClassificationEvaluator` |
| Tuning | `pyspark.ml.tuning` | `ParamGridBuilder`, `CrossValidator`, `TrainValidationSplit` |

Esta actividad utiliza `LinearRegression` como baseline interpretable y `RandomForestRegressor` como modelo principal, con `RegressionEvaluator` para las métricas y un `Pipeline` para el preprocesamiento.

### 1.4 Referencias

Apache Software Foundation. (2026). *MLlib (DataFrame-based): PySpark 4.1.2 documentation*. https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html

Apache Software Foundation. (2026). *RegressionEvaluator: PySpark 4.1.2 documentation*. https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html

Géron, A. (2022). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow: Concepts, tools, and techniques to build intelligent systems* (3rd ed.). O'Reilly Media.

Polak, A. (2023). *Scaling machine learning with Spark: Distributed ML with MLlib, TensorFlow, and PyTorch*. O'Reilly Media.

## 2. Selección de los datos

### 2.0 Reconstrucción compacta de la muestra M (recap de Etapa 2)

> **Nota al profesor:** las próximas cinco celdas reproducen el muestreo estratificado de la **Etapa 2** del proyecto (downcast del esquema, filtros destructivos, imputaciones con auditoría de nulos, construcción del `stratum_id` y `sampleBy` con piso por estrato). Si está familiarizado con esa entrega, puede saltar directamente a la **Sección 2.1**, donde construyo la muestra individual M' a partir de M con ventana exacta. La 2.0 está aquí para que el notebook sea autocontenido y reproducible en Colab.

El bloque se ejecuta en 5 celdas: (a) setup de Spark y rutas; (b) descarga idempotente de los 24 parquets, downcast del esquema y filtros destructivos; (c) imputaciones de seis columnas y auditoría de nulos; (d) construcción del estrato y derivación de `stratum_id`; (e) recálculo del diccionario de fracciones desde primeros principios y extracción de M vía `sampleBy`.

In [1]:
# (a) setup de Spark y rutas
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pathlib import Path
import urllib.request

spark = (SparkSession.builder
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.debug.maxToStringFields", 100)
    .getOrCreate())

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 13:45:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1


In [2]:
# (b) descarga idempotente de los 24 parquets, downcast del esquema y filtros destructivos
CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
YEARS = (2024, 2025)

def fetch(url, target):
    if target.exists():
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, target)

for y in YEARS:
    for m in range(1, 13):
        f = f"yellow_tripdata_{y}-{m:02d}.parquet"
        fetch(f"{CDN_BASE}/trip-data/{f}", DATA_DIR / f)
fetch(f"{CDN_BASE}/misc/taxi_zone_lookup.csv", DATA_DIR / "taxi_zone_lookup.csv")

df_native = (spark.read.option("mergeSchema", "true")
    .parquet(*sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))))
zones = (spark.read.option("header", True).option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv")))

df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

df_filtered = (df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance").between(0, 200))
    .filter(F.col("fare_amount").between(0, 1000))
    .filter(F.col("total_amount").between(0, 1200))
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0))))

n_raw, n_filtered = df_raw.count(), df_filtered.count()
print(f"Crudo: {n_raw:,} | Tras filtros: {n_filtered:,} | Perdida: {(n_raw - n_filtered) / n_raw * 100:.2f}%")
assert (n_raw - n_filtered) / n_raw < 0.15

Crudo: 89,892,322 | Tras filtros: 84,437,138 | Perdida: 6.07%


In [3]:
# (c) imputaciones de seis columnas y auditoría de nulos
df_clean = (df_filtered
    .withColumn("passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
         .otherwise(F.lit(1).cast("byte")))
    .withColumn("cbd_congestion_fee",
        F.when(F.col("cbd_congestion_fee").isNull() | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
               F.lit(0.0).cast("float"))
         .otherwise(F.col("cbd_congestion_fee")))
    .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee", F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), F.lit("F"))))

imputed_cols = ["passenger_count", "cbd_congestion_fee", "congestion_surcharge",
                "Airport_fee", "RatecodeID", "store_and_fwd_flag"]
nulls = df_clean.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols]).first()
assert all((nulls[c] or 0) == 0 for c in imputed_cols), f"Nulos remanentes: {nulls.asDict()}"
print("Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.")

Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.


In [4]:
# (d) construcción del estrato y derivación de `stratum_id`
airport_ids = {r.LocationID for r in zones.filter(F.col("service_zone").isin("Airports", "EWR")).collect()}
unknown_ids = {264, 265}
manhattan_ids = {r.LocationID for r in zones.filter(F.col("Borough") == "Manhattan").collect()} - airport_ids - unknown_ids
outer_ids = {r.LocationID for r in zones.filter(F.col("Borough").isin("Brooklyn", "Queens", "Bronx", "Staten Island")).collect()} - airport_ids - unknown_ids

df_feat = (df_clean
    .withColumn("pu_macro_zone",
        F.when(F.col("PULocationID").isin(sorted(airport_ids)), "airport")
         .when(F.col("PULocationID").isin(sorted(unknown_ids)), "unknown")
         .when(F.col("PULocationID").isin(sorted(manhattan_ids)), "manhattan")
         .when(F.col("PULocationID").isin(sorted(outer_ids)), "outer_borough")
         .otherwise("unknown"))
    .withColumn("payment_group",
        F.when(F.col("payment_type") == 0, "flex")
         .when(F.col("payment_type") == 1, "credit")
         .when(F.col("payment_type") == 2, "cash")
         .otherwise("other"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("day_hour_bucket",
        F.when(F.col("pickup_hour").between(0, 5), "late_night")
         .when(F.col("dow").isin(1, 7), "weekend")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(6, 10), "weekday_am")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(16, 20), "weekday_pm_peak")
         .otherwise("other"))
    .withColumn("trip_distance_bin",
        F.when(F.col("trip_distance") < 1.12, "short")
         .when(F.col("trip_distance") < 12.43, "medium")
         .otherwise("long"))
    .withColumn("is_flex_fare", F.col("payment_type") == 0)
    .withColumn("cbd_period_flag",
        F.when(F.col("tpep_pickup_datetime") < F.lit("2025-01-05"), "pre_cbd").otherwise("post_cbd"))
    .withColumn("stratum_id",
        F.concat_ws("|",
            F.col("pu_macro_zone"), F.col("payment_group"),
            F.col("day_hour_bucket"), F.col("trip_distance_bin"))))
print("Variables de estrato construidas:", sorted(set(df_feat.columns) - set(df_clean.columns)))

Variables de estrato construidas: ['cbd_period_flag', 'day_hour_bucket', 'dow', 'is_flex_fare', 'payment_group', 'pickup_hour', 'pu_macro_zone', 'stratum_id', 'trip_distance_bin']


In [5]:
# (e) recálculo del diccionario de fracciones desde primeros principios y extracción de M vía `sampleBy`
ESTIMATED_M = 5_030_141
N_M_TARGET = 5_000_000
MIN_FLOOR_M = 500

strata_D = (df_feat.groupBy("stratum_id").count()
    .withColumnRenamed("count", "n_D")
    .withColumn("target_n",
        F.least(F.col("n_D"),
                F.greatest(F.lit(MIN_FLOOR_M).cast("long"),
                           F.round(F.lit(N_M_TARGET) * F.col("n_D") / F.lit(n_filtered)).cast("long"))))
    .withColumn("fraction", F.col("target_n") / F.col("n_D")))

fractions = {r["stratum_id"]: float(r["fraction"]) for r in strata_D.select("stratum_id", "fraction").collect()}
assert all(0 < f <= 1.0 for f in fractions.values())

M = df_feat.stat.sampleBy("stratum_id", fractions, seed=42).cache()
n_M = M.count()
print(f"|M| = {n_M:,} (objetivo {N_M_TARGET:,}, esperado ~5.03M)")
assert abs(n_M - ESTIMATED_M) / ESTIMATED_M < 0.02, f"|M| diverge: {n_M:,}"

# Fin de reconstrucción Etapa 1 y 2
M.select("stratum_id", "fare_amount", "trip_distance", "pu_macro_zone", "payment_group").show(5, truncate=False)

|M| = 5,029,725 (objetivo 5,000,000, esperado ~5.03M)
+--------------------------------------+-----------+-------------+-------------+-------------+
|stratum_id                            |fare_amount|trip_distance|pu_macro_zone|payment_group|
+--------------------------------------+-----------+-------------+-------------+-------------+
|manhattan|credit|late_night|medium    |22.6       |5.72         |manhattan    |credit       |
|manhattan|credit|late_night|medium    |35.9       |7.2          |manhattan    |credit       |
|manhattan|credit|late_night|medium    |16.3       |3.67         |manhattan    |credit       |
|outer_borough|credit|late_night|medium|14.2       |2.67         |outer_borough|credit       |
|manhattan|credit|other|short          |8.6        |0.87         |manhattan    |credit       |
+--------------------------------------+-----------+-------------+-------------+-------------+
only showing top 5 rows
